## Estimating the Market Risk of a Stock

In [27]:
import pandas as pd
import numpy as np
from scipy.stats import linregress

In [4]:
#format is google sheets
df = pd.read_csv("../data/googl_sp500_price.csv")
df.head()

,date_gsheets,price_googl,price_sp500
0,01/03/2012 16:00,332.70,1277.06
1,01/04/2012 16:00,334.14,1277.30
2,01/05/2012 16:00,329.51,1281.06
3,01/06/2012 16:00,325.01,1277.81
4,01/09/2012 16:00,311.23,1280.70


In [5]:
df = df.set_index('date_gsheets')
df.head()

,price_googl,price_sp500
date_gsheets,,
01/03/2012 16:00,332.70,1277.06
01/04/2012 16:00,334.14,1277.30
01/05/2012 16:00,329.51,1281.06
01/06/2012 16:00,325.01,1277.81
01/09/2012 16:00,311.23,1280.70


In [6]:
returns_df = df.pct_change(1)
returns_df.head()

,price_googl,price_sp500
date_gsheets,,
01/03/2012 16:00,NaN,NaN
01/04/2012 16:00,0.004328,0.000188
01/05/2012 16:00,-0.013856,0.002944
01/06/2012 16:00,-0.013657,-0.002537
01/09/2012 16:00,-0.042399,0.002262


In [8]:
returns_df.dropna(inplace=True)
returns_df.head()

,price_googl,price_sp500
date_gsheets,,
01/04/2012 16:00,0.004328,0.000188
01/05/2012 16:00,-0.013856,0.002944
01/06/2012 16:00,-0.013657,-0.002537
01/09/2012 16:00,-0.042399,0.002262
01/10/2012 16:00,0.001092,0.008886


In [9]:
new_col_names = ['r_googl', 'r_sp500']
returns_df.columns = new_col_names
returns_df.head()

,r_googl,r_sp500
date_gsheets,,
01/04/2012 16:00,0.004328,0.000188
01/05/2012 16:00,-0.013856,0.002944
01/06/2012 16:00,-0.013657,-0.002537
01/09/2012 16:00,-0.042399,0.002262
01/10/2012 16:00,0.001092,0.008886


In [11]:
deviations = returns_df - returns_df.mean()
deviations.head()

,r_googl,r_sp500
date_gsheets,,
01/04/2012 16:00,0.003468,-0.000331
01/05/2012 16:00,-0.014717,0.002425
01/06/2012 16:00,-0.014517,-0.003056
01/09/2012 16:00,-0.043259,0.001743
01/10/2012 16:00,0.000232,0.008367


In [14]:
print(returns_df['r_googl'].iloc[0] - returns_df['r_googl'].mean())
print(deviations['r_googl'].iloc[0])

0.0034676852055902266
0.0034676852055902266


In [15]:
new_col_names = ['deviations_googl', 'deviations_sp500']
deviations.columns = new_col_names
deviations.head()

,deviations_googl,deviations_sp500
date_gsheets,,
01/04/2012 16:00,0.003468,-0.000331
01/05/2012 16:00,-0.014717,0.002425
01/06/2012 16:00,-0.014517,-0.003056
01/09/2012 16:00,-0.043259,0.001743
01/10/2012 16:00,0.000232,0.008367


In [20]:
product_deviations = deviations['deviations_googl'] * \
                        deviations['deviations_sp500']
print(product_deviations.head(), type(product_deviations))

date_gsheets
01/04/2012 16:00   -0.000001
01/05/2012 16:00   -0.000036
01/06/2012 16:00    0.000044
01/09/2012 16:00   -0.000075
01/10/2012 16:00    0.000002
dtype: float64 <class 'pandas.Series'>


In [21]:
cov_googl_sp500 = product_deviations.sum() / \
                (len(product_deviations) - 1)
print(cov_googl_sp500)

5.916771855624271e-05


In [22]:
var_sp500 = np.var(returns_df['r_sp500'], ddof=1)
print(var_sp500)

5.7409308185174766e-05


In [23]:
beta_googl = cov_googl_sp500 / var_sp500
print(beta_googl)

1.0306293600577132


In [25]:
#there's a python built-in method for covariance
cov_matrix = np.cov(returns_df['r_sp500'], returns_df['r_googl'])
print(cov_matrix)

[[5.74093082e-05 5.91677186e-05]
 [5.91677186e-05 1.93387833e-04]]


In [26]:
covariance = cov_matrix[0][1]
print(covariance)

5.916771855624269e-05


In [28]:
#r_googl is the dependent variable
#r_sp500 is the independent variable
linregress(y=returns_df['r_googl'], x=returns_df['r_sp500'])

LinregressResult(slope=np.float64(1.0306293600577128), intercept=np.float64(0.00032551171199368526), rvalue=np.float64(0.5615379258866243), pvalue=np.float64(5.82420174455615e-126), stderr=np.float64(0.03914690681346339), intercept_stderr=np.float64(0.000297209094419293))

In [29]:
beta = linregress(y=returns_df['r_googl'], x=returns_df['r_sp500'])[0]
print(beta)

1.0306293600577128
